# ScreamingFace · discover models and benchmarks

Discover what this ScreamingFace setup can run without mixing discovery with execution. Models and
benchmarks intentionally come from different places:

- executable model IDs come from the configured engine;
- canonical benchmark IDs come from the installed SDK.

This notebook shows both lists and their shared filters, then makes the separate benchmark-loading
boundary explicit.

## Before you run it

Start the local development stack from the repository root:

```bash
cd packages/screamingface/apps/screamingface-engine
./dev.sh
```

Docker is the only prerequisite for the default run. Listing models does not call them, so no
provider credentials are needed. Benchmark source loading is shown later but remains disabled.

## 1 · Configure the engine

In [ ]:
import os

import screamingface as sf

ENGINE_URL = os.environ.get("SCREAMINGFACE_ENGINE_URL", "http://127.0.0.1:4404")
sf.config(engine=ENGINE_URL)

`sf.config(...)` selects the one engine used by model discovery and later model
execution. It validates and stores the HTTP(S) origin but does not make a request itself.

## 2 · List executable models

In [ ]:
model_ids = sf.models.list()
model_ids

`sf.models.list()` asks the configured engine for its current capability registry
and returns executable model IDs in registry order. These are routes the engine knows how to run.

Discovery does not call a model and does not prove that provider credentials are connected. A
listed route can still fail at execution time if its provider is unavailable or unauthenticated.

## 3 · Filter the model IDs

In [ ]:
gemini_models = sf.models.list(query="gemini")
web_search_models = sf.models.list(tools=("web_search",))
first_two_models = sf.models.list(limit=2)

{
    "query=gemini": gemini_models,
    "tools=web_search": web_search_models,
    "limit=2": first_two_models,
}

The filters are small and predictable:

- `query` is a case-insensitive substring match on the model ID;
- `tools` keeps routes that advertise every requested tool; and
- `limit` returns a stable prefix after filtering.

Here, `web_search` means the engine route advertises that named executable capability. It does not
mean the model learned web content during training, and it does not validate provider access.

## 4 · List installed benchmarks

In [ ]:
benchmark_ids = sf.benchmarks.list()
benchmark_ids

`sf.benchmarks.list()` reads the canonical definitions shipped by the installed SDK.
It does not contact the engine or download benchmark rows. Like model discovery, it deliberately
returns plain IDs rather than introducing a second summary or metadata object.

## 5 · Filter the benchmark IDs

In [ ]:
gpqa_matches = sf.benchmarks.list(query="gpqa")
web_research_benchmarks = sf.benchmarks.list(tools=("web_search",))
first_benchmark = sf.benchmarks.list(limit=1)

{
    "query=gpqa": gpqa_matches,
    "tools=web_search": web_research_benchmarks,
    "limit=1": first_benchmark,
}

The parameter names match model discovery, but `tools` describes a different side
of compatibility. For models it means capabilities the route supports; for benchmarks it means
capabilities the benchmark requires from each answer-producing Fusion member.

## 6 · Listing is not loading

A benchmark ID is enough to select a definition, but it is not the dataset.
`sf.benchmarks.load("gpqa@1")` asks the installed definition to fetch and validate its pinned
source, then returns an immutable `sf.Benchmark` ready for execution.

GPQA is gated on Hugging Face. Log in in the researcher process before enabling the next cell:

```bash
huggingface-cli login
```

The Hugging Face token stays with the researcher process. Neither screamingface-engine nor AI
Gateway receives it.

In [ ]:
LOAD_GPQA = False

loaded_gpqa = None
if LOAD_GPQA:
    loaded_gpqa = sf.benchmarks.load("gpqa@1")

loaded_gpqa

The default value performs no dataset request and invents no substitute benchmark.
Set `LOAD_GPQA = True` only after authenticating if you want to materialize and validate all pinned
GPQA cases.

## Recap

- configure one engine with `sf.config(...)`;
- use `sf.models.list(...)` for model routes executable by that deployment;
- use `sf.benchmarks.list(...)` for canonical definitions installed in the SDK;
- both list APIs return plain IDs and accept `query`, `tools`, and `limit`;
- listing a benchmark is local and network-free; loading can fetch its pinned source; and
- discovery performs no Fusion, model, grader, or aggregation work.

Continue to the quickstart to compose and evaluate a Fusion, or the architecture notebook to inspect
the registry and URL4 HTTP boundary.